# Profilierung: MongoDB-Quelle

**Akt 2, zweite Quelle.** Die MongoDB `LetsMeet` (Collection `users`) ist neben der
Excel-Datei die zweite Quelle: Sie enthält ergänzende Profildaten sowie gerichtete Likes und
Nachrichten. Bevor wir daraus etwas machen, schauen wir uns an, was sie wirklich enthält —
welche Felder, welche Typen, welche Verschachtelungen und Mehrfachwerte, und wo sie der
Excel-Quelle widerspricht.

Das Backup läuft im Compose-Service `mongodb_for_lf8`. Die Dienste müssen laufen
(`letsmeet up` bzw. `docker compose up -d`).

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://127.0.0.1:27017/")
db = client["LetsMeet"]
users = db["users"]

print("Dokumente in users:", users.count_documents({}))

## Ist die Collection leer?

Ein leerer Stand ist kein Fehler — sie ist dann einfach noch nicht befüllt. Wir halten fest,
welche Daten **da sind**, nicht was wir erwarten.

In [ ]:
gesamt = users.count_documents({})
print("Dokumente insgesamt:", gesamt)

beispiel = users.find_one({}, {"_id": 0})
if beispiel is not None:
    print("\nFeldstruktur des ersten Dokuments:")
    for feld, wert in beispiel.items():
        print(f"  {feld:<16} {type(wert).__name__:<12} {wert!r}")

In [ ]:
import pandas as pd

docs = list(users.find({}, {"_id": 0}).limit(20))
if docs:
    display(pd.json_normalize(docs).head(10))
else:
    print("Collection leer — keine Dokumente anzuzeigen.")

## Werteprofile je Feld

Für jedes vorhandene Feld zählen wir: Anzahl gesetzt, Anzahl leer, Anzahl eindeutiger Werte und
die ersten Werte. So sehen wir Ausreißer (z. B. `null`-Werte oder abweichende Schreibweisen),
bevor wir mit den Daten arbeiten.

In [ ]:
from collections import Counter

if gesamt:
    felder = set()
    for doc in users.find({}, {"_id": 0}):
        felder.update(doc.keys())

    for feld in sorted(felder):
        werte = [d.get(feld) for d in users.find({}, {"_id": 0, feld: 1})]
        nicht_leer = [w for w in werte if w not in (None, "", [], {})]
        zaehler = Counter(str(w)[:30] for w in nicht_leer)
        print(f"{feld:<16} gesetzt {len(nicht_leer):>4}/{gesamt:<5} eindeutig {len(zaehler):>4}")
        print(f"    häufigste: {zaehler.most_common(3)}")
else:
    print("Collection leer — keine Felder zu profilieren.")

## Eindeutigkeit der E-Mail

Ist `email` eindeutig (case-insensitiv)? Das ist die Voraussetzung dafür, dass wir die
MongoDB-Dokumente den PostgreSQL-Zeilen zuordnen können.

In [ ]:
if gesamt:
    emails = [d["email"] for d in users.find({"email": {"$exists": True}}, {"_id": 0, "email": 1})]
    print("Dokumente mit email:", len(emails))
    print("davon eindeutig (case-insensitiv):", len({e.lower() for e in emails}))
else:
    print("Collection leer.")

## Befund

Trage hier die beobachteten Zahlen ein (oder fasse sie in der Befundnotiz zusammen):

1. Anzahl Dokumente
2. Welche Felder existieren, welche fehlen?
3. Gibt es leere/`null`-Felder oder Duplikate?
4. Passt die Struktur zu dem, was der Import (Notebook 03) erzeugen wird?

*Deine Antwort:*

1.  
2.  
3.  
4.  